In [5]:
import wikipediaapi
import time
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def download_wikipedia_article(
    article_title: str,
    output_dir: str = "wikipedia_articles",
    language: str = "en",
    user_agent: str = "MyRAGPipeline (contact@example.com)",
    include_html: bool = False
) -> bool:
    """
    Download a Wikipedia article and save it as a text file.
    
    Args:
        article_title: Title of the Wikipedia article
        output_dir: Directory to save the article
        language: Language code (e.g., 'en', 'fr', 'de')
        user_agent: User agent string (required by Wikipedia)
        include_html: If True, saves HTML format; otherwise plain text
    
    Returns:
        True if successful, False otherwise
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    wiki_wiki = wikipediaapi.Wikipedia(
        user_agent=user_agent,
        language=language,
        extract_format=wikipediaapi.ExtractFormat.HTML if include_html else wikipediaapi.ExtractFormat.WIKI
    )
    
    page = wiki_wiki.page(article_title)
    
    if not page.exists():
        logging.warning(f"Article '{article_title}' does not exist.")
        return False
    
    # Prepare filename
    safe_title = "".join(c for c in article_title if c.isalnum() or c in (' ', '_')).rstrip()
    filename = output_path / f"{safe_title}.{'html' if include_html else 'txt'}"
    
    # Save article content
    with open(filename, 'w', encoding='utf-8') as f:
        if include_html:
            f.write(page.text)
        else:
            f.write(f"Title: {page.title}\n")
            f.write(f"URL: {page.fullurl}\n\n")
            f.write("=== SUMMARY ===\n")
            f.write(f"{page.summary}\n\n")
            f.write("=== FULL CONTENT ===\n")
            f.write(page.text)
    
    logging.info(f"Saved: {filename}")
    logging.info(f"  Title: {page.title}")
    logging.info(f"  Summary: {page.summary[:100]}...")
    
    return True

def download_multiple_articles(
    article_titles: list,
    output_dir: str = "wikipedia_articles",
    language: str = "en",
    delay_seconds: float = 1.0
) -> None:
    """
    Download multiple Wikipedia articles with a delay between requests.
    
    Args:
        article_titles: List of article titles to download
        output_dir: Directory to save articles
        language: Language code
        delay_seconds: Delay between API requests (respect Wikipedia's servers)
    """
    for title in article_titles:
        download_wikipedia_article(title, output_dir, language)
        time.sleep(delay_seconds)  # Be respectful to Wikipedia's servers
    
    logging.info(f"Completed. Downloaded {len(article_titles)} articles to '{output_dir}'")

if __name__ == "__main__":
    # Example usage: Download articles relevant to RAG
    articles = [
        "Retrieval-augmented generation",
        "Transformer (deep learning architecture)",
        "Large language model",
        "Embedding",
        "Vector database"
    ]
    
    download_multiple_articles(articles, output_dir="rag_wikipedia_articles", delay_seconds=1.0)

2026-06-08 16:25:15,911 - INFO - Wikipedia: language=en, user_agent: MyRAGPipeline (contact@example.com) Wikipedia-API/0.15.0; https://github.com/martin-majlis/Wikipedia-API/, extract_format=1
2026-06-08 16:25:15,980 - INFO - Request URL: https://en.wikipedia.org/w/api.php?format=json&redirects=1&action=query&prop=info&titles=Retrieval-augmented generation&inprop=protection|talkid|watched|watchers|visitingwatchers|notificationtimestamp|subjectid|url|readable|preload|displaytitle|varianttitles
2026-06-08 16:25:16,352 - INFO - HTTP Request: GET https://en.wikipedia.org/w/api.php?format=json&redirects=1&action=query&prop=info&titles=Retrieval-augmented+generation&inprop=protection%7Ctalkid%7Cwatched%7Cwatchers%7Cvisitingwatchers%7Cnotificationtimestamp%7Csubjectid%7Curl%7Creadable%7Cpreload%7Cdisplaytitle%7Cvarianttitles "HTTP/1.1 429 Too Many Requests"
2026-06-08 16:26:00,760 - INFO - HTTP Request: GET https://en.wikipedia.org/w/api.php?format=json&redirects=1&action=query&prop=info&titl

In [9]:

import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions
from pathlib import Path

db_dir = "chroma_db"
collection_name = "wikipedia_articles"
article_dir = Path("rag_wikipedia_articles")

client = chromadb.PersistentClient(
    path=db_dir
)

embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.get_or_create_collection(
    name=collection_name,
    embedding_function=embedding_function
)

for article_file in sorted(article_dir.glob("*.txt")):
    article_id = article_file.stem
    with open(article_file, "r", encoding="utf-8") as f:
        content = f.read().strip()

    if not content:
        continue

    metadata = {
        "title": article_id,
        "source_path": str(article_file)
    }

    try:
        collection.delete(ids=[article_id])
    except Exception:
        pass

    collection.add(
        ids=[article_id],
        documents=[content],
        metadatas=[metadata]
    )


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.34s/it]


In [29]:
def find_matching_files_for_question(question: str, top_n: int = 5):
    # Query the Chroma collection for the question text
    query_result = collection.query(
        query_texts=[question],
        n_results=top_n,
        include=["metadatas"]
    )
    ids = query_result.get("ids", [[]])[0]
    metadatas = query_result.get("metadatas", [[]])[0]

    # If no matching ids are returned, indicate no files found
    if not ids:
        return False

    file_paths = []
    for metadata, item_id in zip(metadatas, ids):
        # Prefer the source path from metadata; otherwise use the document id
        if isinstance(metadata, dict) and "source_path" in metadata:
            file_paths.append(metadata["source_path"])
        else:
            file_paths.append(item_id)

    return file_paths if file_paths else False

question = input("Enter your question: ")
matched_files = find_matching_files_for_question(question)
matched_files

Batches: 100%|██████████| 1/1 [00:00<00:00, 29.06it/s]


Batches: 100%|██████████| 1/1 [00:00<00:00, 29.06it/s]


['rag_wikipedia_articles/Vector database.txt',
 'rag_wikipedia_articles/Large language model.txt',
 'rag_wikipedia_articles/Transformer deep learning architecture.txt',
 'rag_wikipedia_articles/Retrievalaugmented generation.txt',
 'rag_wikipedia_articles/Embedding.txt']

In [30]:
import ollama
from sentence_transformers import SentenceTransformer
import os
os.environ["HF_HUB_OFFLINE"] = "1"

# Load a small embedding model once
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def cosine_similarity(a, b):
    dot = sum(x*y for x,y in zip(a,b))
    norm_a = sum(x*x for x in a)**0.5
    norm_b = sum(x*x for x in b)**0.5
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0

def is_relevant(question: str, docs: list, threshold: float = 0.5) -> bool:
    """Return True if at least one document is similar enough to the question."""
    if not docs:
        return False
    q_emb = embedder.encode(question)
    for doc in docs:
        d_emb = embedder.encode(doc)
        sim = cosine_similarity(q_emb, d_emb)
        if sim >= threshold:
            return True
    return False

def generate_answer(question: str, context: str, model: str = "gemma3:1b") -> str:
    prompt = f"""Answer the question using only the context below.
If the context doesn't contain enough information, say "I don't know".

Context: {context}

Question: {question}
Answer:"""
    resp = ollama.generate(model=model, prompt=prompt, stream=False)
    return resp['response'].strip()

def rag_with_relevance_check(question: str, local_docs: list, threshold: float = 0.5):
    """Use local docs only if relevant; otherwise return a message."""
    if is_relevant(question, local_docs, threshold):
        print("✓ Documents are relevant. Generating answer from local context.")
        context = "\n\n---\n\n".join(local_docs[:2])  # use top 2 docs
        answer = generate_answer(question, context)
    else:
        print("✗ Documents are NOT relevant to the question.")
        answer = "I cannot answer this question because the retrieved documents are not relevant."
    return answer

answer = rag_with_relevance_check(question, matched_files)


2026-06-08 17:36:23,026 - INFO - No device provided, using cpu
2026-06-08 17:36:23,421 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-06-08 17:36:23,550 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
2026-06-08 17:36:23,707 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-06-08 17:36:23,780 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-06-08 17:36:23,782 - INFO - Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
2026-06-08 17

2026-06-08 17:36:23,026 - INFO - No device provided, using cpu
2026-06-08 17:36:23,421 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-06-08 17:36:23,550 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
2026-06-08 17:36:23,707 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-06-08 17:36:23,780 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-06-08 17:36:23,782 - INFO - Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
2026-06-08 17

✗ Documents are NOT relevant to the question.
